# SYNCEDSTRUCTURE
### **A Class to handle, crossreference and annotate Structural Proteomics Data**

Credits:
LICENCES:

In [1]:
### <DEPENDENCIES and SyncedStructure Methods display> 
import re, requests, io, copy, os, pickle
import pandas as pd
import seaborn as sns
from synced_structure import SyncedStructure
from viz3d import side_by_side

def get_uniprot(uid):
    u = str(uid).strip().upper()
    if not re.match(r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})(-\d+)?$", u):
        raise ValueError()
    buf = io.StringIO(urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{u}.fasta").read().decode())
    return u, "".join(buf.read().splitlines()[1:])


pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

synced_structure_options = dir(SyncedStructure)
d = [x for x in synced_structure_options if x.startswith("__")]
h = [x for x in synced_structure_options
    if x.startswith("_") and not x.startswith("__")]
e = [x for x in synced_structure_options if not x.startswith("_")]
for g in (d, h, e):
  if g:
    [print(g[i : i + 4]) for i in range(0, len(g), 4)]
    print()
### <\DEPENDENCIES and SyncedStructure Methods display>

['__class__', '__contains__', '__delattr__', '__dict__']
['__dir__', '__doc__', '__eq__', '__firstlineno__']
['__format__', '__ge__', '__getattribute__', '__getitem__']
['__getstate__', '__gt__', '__hash__', '__init__']
['__init_subclass__', '__iter__', '__le__', '__len__']
['__lt__', '__module__', '__ne__', '__new__']
['__reduce__', '__reduce_ex__', '__repr__', '__setattr__']
['__sizeof__', '__static_attributes__', '__str__', '__subclasshook__']
['__weakref__']

['_apply_chain_options', '_atom_contacts', '_atom_pair_maps', '_auth_of']
['_block_entries', '_build_chains', '_ca_records', '_chain_colors']
['_chain_contact_partners', '_chain_description_map', '_chain_entity_map', '_chain_geometry']
['_checked_matrix', '_clean_val', '_compute_interface', '_compute_local_alignment']
['_coords_touch', '_draw_pair', '_entity_sequences', '_external_refs_for_entity']
['_gray_matrix', '_heavy_atoms', '_hydrophobicity', '_interactive_view']
['_keep_mapped', '_label_of', '_local_alignment', '_make_

# Human Timeless Homolog Interfaces

In [2]:
import requests

upid_list = ["Q9UNS1"]
payload = {
    "query": {
        "type": "terminal",
        "service": "text",
        "parameters": {
            "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
            "operator": "in",
            "value": upid_list
        }
    },
    "return_type": "entry",
    "request_options": {"paginate": {"start": 0, "rows": 250}}
}
### r = requests.post("https://search.rcsb.org/rcsbsearch/v2/query", json=payload)
### print([x["identifier"] for x in r.json().get("result_set", [])])

In [3]:
import os, pickle
import pandas as pd
pdb_ids = ['4XHT', '4XHU', '4XHW', '5MQI', '6T9Q', '6TAZ', '7PFO', '7PLO', '8B9D', '9O34']
TIM_INTERFACES_PATH = "tutorial-data/TIM/TIM-interfaces-raw.pkl"

try:
    if os.path.exists(TIM_INTERFACES_PATH):
        with open(TIM_INTERFACES_PATH, "rb") as f:
            synced_list = pickle.load(f)
    else:
        raise FileNotFoundError
except (EOFError, FileNotFoundError):
    synced_list = [SyncedStructure.from_rcsb_id(pid) for pid in pdb_ids]
    os.makedirs("tutorial-data/TIM", exist_ok=True)
    with open(TIM_INTERFACES_PATH, "wb") as f:
        pickle.dump(synced_list, f)

sync_summaries = []
for pdb_id, sync in zip(pdb_ids, synced_list):
    summary=sync.summary()
    summary['pdb_id']=pdb_id
    sync_summaries.append(summary)
sync_summaries = [sync.summary() for sync in synced_list]

sync_df = pd.concat(sync_summaries, ignore_index=True)

In [4]:
print(sync_df)

    chain_id auth_chain_id entity_id                     description external_ref  sequence_length  reference_length  mapped_residues  mapping_fraction
0          A             A         1        Protein timeless homolog       Q9UNS1               94               103               94               1.0
1          B             C         1        Protein timeless homolog       Q9UNS1               94               103               94               1.0
2          C             B         1        Protein timeless homolog       Q9UNS1               92               103               92               1.0
3          D             D         1        Protein timeless homolog       Q9UNS1               89               103               89               1.0
4          A             C         1  Poly [ADP-ribose] polymerase 1       P09874              333               359              333               1.0
..       ...           ...       ...                             ...          ...       

In [5]:
print(sync_df[sync_df['external_ref']==upid_list[0]])

   chain_id auth_chain_id entity_id                                        description external_ref  sequence_length  reference_length  mapped_residues  mapping_fraction
0         A             A         1                           Protein timeless homolog       Q9UNS1               94               103               94               1.0
1         B             C         1                           Protein timeless homolog       Q9UNS1               94               103               94               1.0
2         C             B         1                           Protein timeless homolog       Q9UNS1               92               103               92               1.0
3         D             D         1                           Protein timeless homolog       Q9UNS1               89               103               89               1.0
5         B             D         2                           Protein timeless homolog       Q9UNS1               83               103               8